## Merging the FIFA Rankings

In [1]:
import pandas as pd

In [ ]:
# 1. Load the processed datasets
df_matches = pd.read_csv('../data/processed/international_matches.csv')
df_fifa = pd.read_csv('../data/processed/fifa_rankings.csv')

In [14]:
# 2. Rename 'country' to 'host_country' in matches so it doesn't collide
df_matches.rename(columns={'country': 'host_country'}, inplace=True)

In [15]:
# 3. Ensure dates are actual Datetime objects
df_matches['date'] = pd.to_datetime(df_matches['date'])
df_fifa['rank_date'] = pd.to_datetime(df_fifa['rank_date'])

In [18]:
# 4. Sort both dataframes by date (merge_asof REQUIRES sorted data)
df_matches = df_matches.sort_values('date').reset_index(drop=True)
df_fifa = df_fifa.sort_values('rank_date').reset_index(drop=True)

In [ ]:
# 5. HOME TEAM MERGE (Bulletproof method)

# We create a temporary FIFA dataframe just for the Home Team
df_fifa_home = df_fifa[['rank_date', 'country', 'fifa_rank', 'fifa_points']].copy()
df_fifa_home.rename(columns={'country': 'home_team', 'fifa_rank': 'home_fifa_rank', 'fifa_points': 'home_fifa_points'}, inplace=True)

df_matches = pd.merge_asof(
    df_matches, 
    df_fifa_home, 
    left_on='date', 
    right_on='rank_date', 
    by='home_team', 
    direction='backward'
)
df_matches.drop(columns=['rank_date'], inplace=True) # clean up

In [20]:
# 6. AWAY TEAM MERGE (Bulletproof method)

# We create a temporary FIFA dataframe just for the Away Team
df_fifa_away = df_fifa[['rank_date', 'country', 'fifa_rank', 'fifa_points']].copy()
df_fifa_away.rename(columns={'country': 'away_team', 'fifa_rank': 'away_fifa_rank', 'fifa_points': 'away_fifa_points'}, inplace=True)

df_matches = pd.merge_asof(
    df_matches, 
    df_fifa_away, 
    left_on='date', 
    right_on='rank_date', 
    by='away_team', 
    direction='backward'
)
df_matches.drop(columns=['rank_date'], inplace=True) # clean up

In [21]:
# 7. Look at the result!
display(df_matches[['date', 'home_team', 'away_team', 'home_fifa_rank', 'away_fifa_rank']].tail(10))

,date,home_team,away_team,home_fifa_rank,away_fifa_rank
30759,2026-06-26,Cape Verde,Saudi Arabia,NaN,56.0
30760,2026-06-26,Uruguay,Spain,14.0,8.0
30761,2026-06-26,Norway,France,46.0,2.0
30762,2026-06-26,Senegal,Iraq,18.0,55.0
30763,2026-06-27,DR Congo,Uzbekistan,59.0,62.0
30764,2026-06-27,Panama,England,43.0,5.0
30765,2026-06-27,Jordan,Argentina,68.0,1.0
30766,2026-06-27,Algeria,Austria,44.0,25.0
30767,2026-06-27,Colombia,Portugal,12.0,6.0
30768,2026-06-27,Croatia,Ghana,9.0,64.0


In [22]:
df_matches.isna().sum()

date                   0
home_team              0
away_team              0
home_score            44
away_score            44
tournament             0
city                   0
country_x              0
neutral                0
rank_date_x         2000
country_y           2000
fifa_rank_x         2003
fifa_points_x       2000
rank_date_y         2000
host_country        2000
fifa_rank_y         2003
fifa_points_y       2000
home_fifa_rank      2003
home_fifa_points    2000
away_fifa_rank      2193
away_fifa_points    2187
dtype: int64

## 🧹 Removing Future Matches and Completing the Final Data Merges

The **Kaggle Match Results** dataset includes not only historical matches but also **scheduled fixtures for the 2026 FIFA World Cup**. While these future matches are useful for prediction later, they **must not be included during model training** because they have not been played yet and therefore do not contain match outcomes.

Training a machine learning model on matches without actual results would introduce incomplete and misleading data. To avoid this, we first remove all future fixtures, ensuring that the training dataset contains only completed matches with known outcomes.

After cleaning the dataset, we perform the final two merges:

- **EA Sports Team Ratings**
- **Transfermarkt Team Market Values**

Unlike the FIFA Rankings dataset, these two datasets are organized by **year**, making the merge process much simpler. Instead of matching by exact dates, we only need to match:

- **Country (Team Name)**
- **Match Year**

This approach correctly assigns each team's EA rating and market value corresponding to the year in which the match was played, providing additional features for the prediction model.

In [23]:
# 1. Filter out future matches (If a match hasn't happened, it has no score!)
df_matches = df_matches.dropna(subset=['home_score', 'away_score']).copy()

In [24]:
# 2. Extract the year from the match date so we can link EA and TM data
df_matches['match_year'] = df_matches['date'].dt.year

In [25]:
# 3. Load the processed EA and TM datasets
df_ea = pd.read_csv('../data/processed/ea_squad_strength.csv')
df_tm = pd.read_csv('../data/processed/tm_squad_value.csv')

In [26]:
# 4. Merge EA Sports Data (Skill)

# Home Team
df_ea_home = df_ea.rename(columns={'country': 'home_team', 'year': 'match_year', 'ea_squad_score': 'home_ea_score'})
df_matches = pd.merge(df_matches, df_ea_home, on=['home_team', 'match_year'], how='left')

# Away Team
df_ea_away = df_ea.rename(columns={'country': 'away_team', 'year': 'match_year', 'ea_squad_score': 'away_ea_score'})
df_matches = pd.merge(df_matches, df_ea_away, on=['away_team', 'match_year'], how='left')

In [27]:
# 5. Merge Transfermarkt Data (Money)

# Home Team
df_tm_home = df_tm.rename(columns={'country': 'home_team', 'year': 'match_year', 'tm_squad_value_eur': 'home_tm_value'})
df_matches = pd.merge(df_matches, df_tm_home, on=['home_team', 'match_year'], how='left')

# Away Team
df_tm_away = df_tm.rename(columns={'country': 'away_team', 'year': 'match_year', 'tm_squad_value_eur': 'away_tm_value'})
df_matches = pd.merge(df_matches, df_tm_away, on=['away_team', 'match_year'], how='left')

In [28]:
# 6. Look at a massive recent match (e.g., the 2022 World Cup Final!)
wc_final = df_matches[(df_matches['home_team'] == 'Argentina') & 
                      (df_matches['away_team'] == 'France') & 
                      (df_matches['match_year'] == 2022)]

# Let's display our final master columns
cols_to_show = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 
                'home_fifa_rank', 'away_fifa_rank', 'home_ea_score', 'away_ea_score', 
                'home_tm_value', 'away_tm_value']

In [29]:
display(wc_final[cols_to_show])

,date,home_team,away_team,home_score,away_score,home_fifa_rank,away_fifa_rank,home_ea_score,away_ea_score,home_tm_value,away_tm_value
27284,2022-12-18,Argentina,France,3.0,3.0,3.0,4.0,83.7,85.05,816000000.0,1.358000e+09


## 📊 What Does a Single Match Tell Us?

A single row in our dataset contains far more than just the final score—it captures the context behind the match.

Consider the **2022 FIFA World Cup Final** between **France** and **Argentina**:

- **France** had a higher **EA Sports Team Rating** (85.05 vs. 83.70).
- Their squad was valued at **more than €500 million higher** than Argentina's according to Transfermarkt.
- Despite these apparent advantages, the match ended **3–3 after extra time**, with Argentina eventually winning on penalties.

This example highlights an important principle of football analytics: **stronger teams do not always win**.

While features such as team ratings, market values, FIFA rankings, and historical performance provide valuable information, football matches are influenced by many unpredictable factors, including tactics, player form, injuries, and moments of individual brilliance.

> **Key Takeaway:**  
> Machine learning models do not rely on a single statistic to make predictions. Instead, they analyze the combined influence of multiple features to identify patterns and estimate the probability of different outcomes. This is why incorporating diverse data sources helps build a more accurate and robust prediction model.

In [30]:
# 7. Save the final dataset

df_matches.to_csv('../data/final/master_training_data.csv', index=False)
print("Master Training Dataset saved successfully!")
print(f"Total Matches ready for Machine Learning: {len(df_matches)}")

Master Training Dataset saved successfully!
Total Matches ready for Machine Learning: 30941
